# Notebook for testing functionality of individual files

In [1]:
from calculator.pricing import blended_price_per_1m, api_monthly_cost_usd, FLASH_PRICES
from calculator.benchmarks import self_host_config_for
from calculator.decision import evaluate_workload
from calculator.models import WorkloadInputs

In [6]:
## Pricing sanity checks

# Flash blended examples from the paper
pricing = FLASH_PRICES["<=200k"]

# Simple extraction: alpha=0.60 beta=0.40 gamma=0.00 -> expected 0.33
p = blended_price_per_1m(0.60, 0.40, 0.00, pricing)
assert abs(p - 0.33) < 1e-9, p

# Standard chat: alpha=0.40 beta=0.30 gamma=0.30 -> expected 0.45
p = blended_price_per_1m(0.40, 0.30, 0.30, pricing)
assert abs(p - 0.45) < 1e-9, p

# Heavy reasoning: alpha=0.25 beta=0.25 gamma=0.50 -> expected 0.54
p = blended_price_per_1m(0.25, 0.25, 0.50, pricing)
assert abs(p - 0.54) < 1e-9, p

print("Pricing tests passed.")

AssertionError: 0.5375

In [3]:
## API monthly cost check

# Table 6 uses simple profile alpha=0.50 beta=0.50 gamma=0.00 -> P = 0.375
p = blended_price_per_1m(0.50, 0.50, 0.00, FLASH_PRICES["<=200k"])
assert abs(p - 0.375) < 1e-12, p

# 30M/day => 30 * 30M / 1e6 * 0.375 = 337.5
c = api_monthly_cost_usd(30_000_000, p)
assert abs(c - 337.5) < 1e-9, c

print("API monthly cost test passed.")

API monthly cost test passed.


In [4]:
## Benchmarks config mapping

cfg_small = self_host_config_for("small")
assert cfg_small.gpu_name == "RTX4090"
assert cfg_small.gpus_per_replica == 1
assert cfg_small.throughput_tok_per_s > 0

cfg_large = self_host_config_for("large")
assert cfg_large.gpus_per_replica == 2

print("Benchmark config tests passed.")

Benchmark config tests passed.


In [5]:
## Decision function smoke test

w = WorkloadInputs(
    total_tokens_per_day=1_000_000,
    alpha_in=0.50,
    beta_out=0.50,
    gamma_think=0.00,
    model_class="small",
    utilisation_target=0.75,
    overhead_rate=0.30,
    context_bucket="<=200k",
)

res = evaluate_workload(w)
assert res.api_monthly_usd > 0
assert res.self_host_monthly_usd > 0
assert res.replicas_required >= 1
assert res.recommendation in ("API", "SELF_HOST")
print("Decision smoke test passed.")

Decision smoke test passed.
